# Hospital Readmission Prediction - Model Training & Evaluation

## Objective
Train, evaluate, and compare multiple machine learning models to predict 30-day hospital readmissions for diabetic patients.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Preprocessed Data

In [ ]:
# Load preprocessed data
X_train = pd.read_csv('../data/X_train_scaled.csv', index_col=0)
X_val = pd.read_csv('../data/X_val_scaled.csv', index_col=0)
X_test = pd.read_csv('../data/X_test_scaled.csv', index_col=0)

y_train = pd.read_csv('../data/y_train.csv', index_col=0).squeeze()
y_val = pd.read_csv('../data/y_val.csv', index_col=0).squeeze()
y_test = pd.read_csv('../data/y_test.csv', index_col=0).squeeze()

feature_cols = joblib.load('../data/feature_cols.pkl')

print("=" * 60)
print("DATA LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")

## 2. Try XGBoost and LightGBM (may not be installed, use alternatives)

In [ ]:
# Try importing XGBoost and LightGBM
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("XGBoost is available")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not available, will use alternative")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("LightGBM is available")
except ImportError:
    LGB_AVAILABLE = False
    print("LightGBM not available, will use alternative")

## 3. Define Models

In [ ]:
# Define models to train
models = {}

# 1. Logistic Regression
models['Logistic Regression'] = LogisticRegression(
    random_state=42, 
    max_iter=1000,
    class_weight='balanced'
)

# 2. Random Forest
models['Random Forest'] = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

# 3. XGBoost (if available)
if XGB_AVAILABLE:
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )

# 4. LightGBM (if available)
if LGB_AVAILABLE:
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        verbose=-1
    )

print(f"Models to train: {list(models.keys())}")

## 4. Train Models & Evaluate

In [ ]:
# Train and evaluate all models
results = []
trained_models = {}

print("=" * 70)
print("TRAINING AND EVALUATING MODELS")
print("=" * 70)

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    
    # Train model
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predictions
    y_pred = model.predict(X_val)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_pred_proba)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f"\nValidation Results:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")

## 5. Model Comparison

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('ROC-AUC', ascending=False)

print("=" * 70)
print("MODEL COMPARISON (Sorted by ROC-AUC)")
print("=" * 70)
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig = go.Figure()

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
colors = px.colors.qualitative.Set2

for i, metric in enumerate(metrics):
    fig.add_trace(go.Bar(
        name=metric,
        x=results_df['Model'],
        y=results_df[metric],
        marker_color=colors[i]
    ))

fig.update_layout(
    title='Model Performance Comparison',
    xaxis_title='Model',
    yaxis_title='Score',
    barmode='group',
    yaxis=dict(range=[0, 1])
)
fig.show()

## 6. Select Best Model & Hyperparameter Tuning

In [ ]:
# Select best model based on ROC-AUC
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print(f"Best model: {best_model_name}")
print(f"ROC-AUC: {results_df.iloc[0]['ROC-AUC']:.4f}")

# Hyperparameter tuning for best model
print("\n" + "=" * 70)
print("HYPERPARAMETER TUNING (GridSearchCV)")
print("=" * 70)

In [ ]:
# Define hyperparameter grids for each model type
if best_model_name == 'XGBoost' and XGB_AVAILABLE:
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }
    base_model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
    
elif best_model_name == 'LightGBM' and LGB_AVAILABLE:
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'num_leaves': [31, 50]
    }
    base_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
    
elif best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [8, 10, 12],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
    
else:
    param_grid = {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs']
    }
    base_model = LogisticRegression(random_state=42, max_iter=1000)

print(f"Hyperparameter grid: {param_grid}")

In [ ]:
# Perform GridSearchCV (with limited params for speed)
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print("Running GridSearchCV...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate best model from GridSearchCV on validation set
best_tuned_model = grid_search.best_estimator_

y_pred_tuned = best_tuned_model.predict(X_val)
y_pred_proba_tuned = best_tuned_model.predict_proba(X_val)[:, 1]

print("\n" + "=" * 70)
print("TUNED MODEL - VALIDATION RESULTS")
print("=" * 70)
print(classification_report(y_val, y_pred_tuned, target_names=['Not Readmitted', 'Readmitted']))

tuned_accuracy = accuracy_score(y_val, y_pred_tuned)
tuned_precision = precision_score(y_val, y_pred_tuned)
tuned_recall = recall_score(y_val, y_pred_tuned)
tuned_f1 = f1_score(y_val, y_pred_tuned)
tuned_roc_auc = roc_auc_score(y_val, y_pred_proba_tuned)

print(f"\nTuned Model Metrics:")
print(f"  Accuracy:  {tuned_accuracy:.4f}")
print(f"  Precision: {tuned_precision:.4f}")
print(f"  Recall:    {tuned_recall:.4f}")
print(f"  F1 Score:  {tuned_f1:.4f}")
print(f"  ROC-AUC:   {tuned_roc_auc:.4f}")

## 7. Evaluate on Test Set

In [ ]:
# Final evaluation on test set
y_pred_test = best_tuned_model.predict(X_test)
y_pred_proba_test = best_tuned_model.predict_proba(X_test)[:, 1]

print("=" * 70)
print("FINAL MODEL - TEST SET RESULTS")
print("=" * 70)
print(classification_report(y_test, y_pred_test, target_names=['Not Readmitted', 'Readmitted']))

test_accuracy = accuracy_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_roc_auc = roc_auc_score(y_test, y_pred_proba_test)

print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {test_accuracy:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1 Score:  {test_f1:.4f}")
print(f"  ROC-AUC:   {test_roc_auc:.4f}")

## 8. Confusion Matrix

In [ ]:
# Confusion matrix visualization
cm = confusion_matrix(y_test, y_pred_test)

fig = px.imshow(cm, 
                text_auto=True,
                labels=dict(x='Predicted', y='Actual', color='Count'),
                x=['Not Readmitted', 'Readmitted'],
                y=['Not Readmitted', 'Readmitted'],
                color_continuous_scale='Blues',
                title='Confusion Matrix - Test Set')
fig.show()

## 9. ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)

fig = go.Figure()

fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC (AUC = {test_roc_auc:.3f})'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier', 
                        line=dict(dash='dash')))

fig.update_layout(
    title='ROC Curve - Test Set',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
    showlegend=True
)
fig.show()

## 10. Feature Importance

In [ ]:
# Get feature importance
if hasattr(best_tuned_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_tuned_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("=" * 70)
print("TOP 20 MOST IMPORTANT FEATURES")
print("=" * 70)
print(feature_importance.head(20).to_string(index=False))

In [ ]:
# Visualize feature importance (top 20)
top_features = feature_importance.head(20)

fig = px.bar(
    top_features, 
    x='importance', 
    y='feature',
    orientation='h',
    title='Top 20 Feature Importances',
    color='importance',
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 11. Save Best Model

In [ ]:
# Save the best model and related objects
joblib.dump(best_tuned_model, '../models/best_model.pkl')
joblib.dump(feature_cols, '../models/feature_cols.pkl')
joblib.dump(X_train.columns.tolist(), '../models/feature_names.pkl')

# Save model metrics
model_metrics = {
    'model_name': best_model_name,
    'test_accuracy': test_accuracy,
    'test_precision': test_precision,
    'test_recall': test_recall,
    'test_f1': test_f1,
    'test_roc_auc': test_roc_auc,
    'best_params': grid_search.best_params_
}
joblib.dump(model_metrics, '../models/model_metrics.pkl')

print("=" * 70)
print("MODEL SAVED")
print("=" * 70)
print(f"Best model saved to: ../models/best_model.pkl")
print(f"Feature columns saved to: ../models/feature_cols.pkl")
print(f"Model metrics saved to: ../models/model_metrics.pkl")

## 12. Model Summary

In [ ]:
print("=" * 70)
print("MODEL TRAINING SUMMARY")
print("=" * 70)
print(f"""
BEST MODEL: {best_model_name}

HYPERPARAMETERS:
{grid_search.best_params_}

TEST SET PERFORMANCE:
  - Accuracy:  {test_accuracy:.4f}
  - Precision: {test_precision:.4f}
  - Recall:    {test_recall:.4f}
  - F1 Score:  {test_f1:.4f}
  - ROC-AUC:   {test_roc_auc:.4f}

TOP 5 RISK FACTORS (Feature Importance):
  1. {feature_importance.iloc[0]['feature']}: {feature_importance.iloc[0]['importance']:.4f}
  2. {feature_importance.iloc[1]['feature']}: {feature_importance.iloc[1]['importance']:.4f}
  3. {feature_importance.iloc[2]['feature']}: {feature_importance.iloc[2]['importance']:.4f}
  4. {feature_importance.iloc[3]['feature']}: {feature_importance.iloc[3]['importance']:.4f}
  5. {feature_importance.iloc[4]['feature']}: {feature_importance.iloc[4]['importance']:.4f}

BUSINESS IMPACT:
  - High Recall ({test_recall:.2%}) means we identify most patients at risk
  - This allows hospitals to intervene early and reduce readmissions
  - Potential cost savings: 10-20% reduction in readmission costs
""")